# Урок 3.3. Оценка качества RAG: golden-датасет, судья и его калибровка

**Модуль 3 «RAG: Retrieval-Augmented Generation на практике» · Курс «LLM для Python-разработчиков»**

У сервиса из урока 3.2 латентность разложена по полочкам — `search_ms / queue_ms / llm_ms`. А качество ответов мы до сих пор оцениваем взглядом: «вроде правильно». Так не поймать регрессию: поменяли top_k, промпт или чанкинг — стало лучше или хуже? Сегодня строим **оценку качества как тесты**: golden-датасет, метрики для каждого этапа конвейера, LLM-судья, которому мы сначала *измерим* доверие, — и RAGAS, который преподнесёт главный сюрприз урока.

## 🎯 Цели урока

После этого урока вы сможете:

- оценивать RAG **по этапам**: ретривер отдельно (hit@k, MRR — без LLM), генератор отдельно (faithfulness, coverage, отказы);
- собрать **golden-датасет** с атомарными эталонными фактами и вопросами-ловушками;
- построить **LLM-судью из бинарных проверок** — и откалибровать его на подложных парах, прежде чем верить;
- увидеть, как метрики ловят **настоящую галлюцинацию** и **настоящую регрессию** — обе пойманы этими самыми метриками при подготовке урока;
- запустить **RAGAS на полностью локальном стеке** — и объяснить, почему он поставил 0.0 идеальному ответу;
- гонять метрики как **регрессионные тесты** при каждом изменении пайплайна.

## План

1. Проблема: «вроде отвечает правильно»
2. Golden-датасет: вопросы, факты, ловушки
3. Этап 1 — ретривер: метрики без LLM
4. Судья и его калибровка
5. Этап 2 — генератор: faithfulness, coverage, отказы
6. Answer relevancy: идея обратных вопросов
7. RAGAS: фреймворк и его сюрприз
8. Метрики как регрессионные тесты

⏱ **Время:** ~90–120 минут (ячейки с полными прогонами работают по 1–2 минуты).

> 💡 Понадобятся: Qdrant с коллекцией урока 3.2 (ячейка настройки проверит и при необходимости переиндексирует сама), Ollama с `qwen2.5:3b`. RAGAS опционален: секция 7 запустится, если он установлен, — и версии там важны, об этом в самой секции.

## 1. Проблема: «вроде отвечает правильно»

В уроке 3.1 мы хвалили RAG глазами: ответ про больничные выглядел верным, офтопик — отбитым. Для первого запуска это нормально. Но сервис живёт дольше первого запуска, и каждое изменение — новый промпт, другой top_k, переиндексация, свежая модель — молча сдвигает качество. В какую сторону? Без чисел вы узнаете об этом от пользователей.

С латентностью мы это уже проходили: в 3.2 появился `search_ms / queue_ms / llm_ms`, и вопрос «почему медленно?» перестал быть загадкой. Сегодня то же самое произойдёт с качеством.

Ключевая идея — **оценивать конвейер по этапам**. RAG ломается в двух разных местах, и лечатся они разными инструментами:

```text
вопрос ──> [ РЕТРИВЕР ] ──чанки──> [ ГЕНЕРАТОР ] ──> ответ
               │                        │
               │ нашёл не то?           │ выдумал? недосказал?
               │ hit@k, MRR             │ faithfulness, coverage,
               │ (без LLM, мс)          │ отказы (LLM-судья, секунды)
```

Если мерить только финальный ответ, диагноз теряется: плохой ответ при промахе ретривера — вина поиска, а не модели. Урок 3.1 уже научил этому на глаз (диагностика по источникам) — теперь формализуем в числа.

| этап | метрика | чем считается | что ловит |
|---|---|---|---|
| ретривер | hit@k | точное сравнение doc_id | «нужный документ вообще не найден» |
| ретривер | MRR | точное сравнение + позиция | «найден, но на дне выдачи» |
| генератор | faithfulness | LLM-судья | выдумки: утверждения не из контекста |
| генератор | coverage | LLM-судья | недосказанность: эталонные факты потеряны |
| генератор | отказы | сравнение строк | «я не знаю» там, где надо (и где не надо) |
| сквозная | answer relevancy | LLM + эмбеддинги | ответ не на тот вопрос |

Обратите внимание на колонку «чем считается»: метрики ретривера **детерминированные и бесплатные** — их можно гонять при каждом коммите. Судья — медленный и сам может ошибаться, поэтому у него будет отдельная глава.

## 2. Golden-датасет: вопросы, факты, ловушки

Все метрики считаются относительно **golden-датасета** — набора вопросов с заранее известными правильными ответами. Наш лежит в [`eval/golden.json`](eval/golden.json), 13 вопросов. Анатомия записи:

```json
{
  "question":        "Сколько компания доплачивает в первые дни болезни?",
  "expected_docs":   ["bolnichny"],          // где лежит ответ (для ретривера)
  "answerable":      true,
  "reference":       "До 100% оклада первые 10 дней болезни при стаже от года.",
  "reference_facts": [                        // эталон, разбитый на АТОМЫ
    "Доплата — до 100% оклада",
    "Доплата действует первые 10 дней болезни",
    "Условие — стаж в компании больше года"
  ]
}
```

Три решения, которые делают датасет полезным:

1. **Факты атомарные.** Не один эталонный абзац, а список коротких самостоятельных утверждений. Зачем — станет ясно в главе про судью: маленькой модели по силам только бинарные вопросы «этот факт донесён? да/нет».
2. **Есть неотвечаемые вопросы** (`answerable: false`). Система, которая отвечает на всё, страшнее системы, которая отвечает на половину: без вопросов-ловушек вы не узнаете, что она выдумывает. Самая коварная ловушка — **тематически близкая**: «Сколько платят за переработки в выходные?» — про деньги и графики в базе много, а ответа на этот вопрос нет.
3. **Есть канарейка** — вопрос «Упал прод, что делать в первую очередь?», на котором лексический поиск гарантированно промахивается (модуль 2 это доказал: в документах — «инциденты», синонимов TF-IDF не знает). Известная слабость должна быть видна в отчёте постоянно — исчезнет из отчёта, значит, починили.

Откуда брать вопросы в реальном проекте: из логов реальных запросов пользователей, тикетов поддержки и FAQ — не из головы автора системы. Автор задаёт вопросы «как надо», пользователи — как получится.

In [ ]:
import json
import os
import re
import subprocess
import sys
import time
from pathlib import Path

import httpx

sys.path.insert(0, os.getcwd())

QDRANT_URL = os.getenv("QDRANT_URL", "http://127.0.0.1:6333")   # 127.0.0.1! (урок 3.2)
OLLAMA_URL = os.getenv("OLLAMA_URL", "http://127.0.0.1:11434")
MODEL = os.getenv("LLM_MODEL", "qwen2.5:3b")
COLLECTION = "vectorika_rag_tfidf"
STATE_PATH = Path("service/tfidf_state.json")


def check(label, probe):
    try:
        detail = probe()
        print(f"[ OK ] {label}: {detail}")
        return True
    except Exception as error:
        print(f"[SKIP] {label}: {type(error).__name__} - зависимые ячейки пропустятся")
        return False


def _qdrant():
    return "v" + httpx.get(QDRANT_URL, timeout=3).json()["version"]


def _ollama():
    models = [m["name"] for m in
              httpx.get(f"{OLLAMA_URL}/api/tags", timeout=3).json()["models"]]
    assert MODEL in models, f"модель {MODEL} не скачана"
    return MODEL


def _ragas():
    import ragas  # noqa: F401
    from langchain_ollama import ChatOllama  # noqa: F401
    return f"v{ragas.__version__}"


QDRANT_AVAILABLE = check("Qdrant", _qdrant)
OLLAMA_AVAILABLE = check("Ollama и модель", _ollama)
RAGAS_AVAILABLE = check("ragas + langchain-ollama (опционально, секция 7)", _ragas)

# коллекция урока 3.2 на месте? если нет - переиндексируем (service/index_kb.py)
if QDRANT_AVAILABLE:
    have = {c["name"] for c in
            httpx.get(f"{QDRANT_URL}/collections", timeout=5).json()["result"]["collections"]}
    if COLLECTION not in have or not STATE_PATH.exists():
        print("коллекции нет - запускаю индексацию...")
        result = subprocess.run([sys.executable, "service/index_kb.py"],
                                capture_output=True, text=True, encoding="utf-8")
        print(result.stdout.strip() or result.stderr.strip())

golden = json.loads(Path("eval/golden.json").read_text(encoding="utf-8"))
answerable = [g for g in golden if g["answerable"]]
print(f"\ngolden-датасет: {len(golden)} вопросов "
      f"({len(answerable)} отвечаемых + {len(golden) - len(answerable)} ловушки)")
print("пример ловушки:", [g["question"] for g in golden if not g["answerable"]][0])

## 3. Этап 1 — ретривер: метрики без единого вызова LLM

Ретривер оценивается точным сравнением: мы знаем, в каком документе лежит ответ (`expected_docs`), и видим, какие документы вернул поиск. Никакой модели — только арифметика:

- **hit@k** — доля вопросов, для которых нужный документ попал в топ-k. `hit@1` — «нужное сразу первым», `hit@3` — «нужное хоть где-то в тройке» (модель получает три чанка, так что hit@3 — это «у генератора вообще был шанс»).
- **MRR** (mean reciprocal rank) — средний `1/позиция` первого правильного документа: первым — 1.0, вторым — 0.5, третьим — 0.33, не нашёлся — 0. Штрафует за «правильный ответ на дне выдачи».

Считаются за миллисекунды, полностью воспроизводимы — идеальный кандидат на прогон при каждом изменении индексации. Меряем наш стек из 3.2 (TF-IDF + Qdrant):

In [ ]:
from service.embedder import TfidfEmbedder

if QDRANT_AVAILABLE:
    from qdrant_client import QdrantClient

    emb = TfidfEmbedder.load(STATE_PATH)
    qdrant = QdrantClient(url=QDRANT_URL)

    def retrieve(question: str, top_k: int = 3) -> list[dict]:
        hits = qdrant.query_points(COLLECTION, query=emb.embed_query(question),
                                   limit=top_k).points
        return [{**hit.payload, "score": hit.score} for hit in hits]

    hit1 = hit3 = 0
    mrr_sum = 0.0
    for item in answerable:
        docs = [c["doc_id"] for c in retrieve(item["question"])]
        expected = set(item["expected_docs"])
        rank = next((i for i, d in enumerate(docs, 1) if d in expected), None)
        hit1 += rank == 1
        hit3 += rank is not None
        mrr_sum += 1 / rank if rank else 0
        mark = "@1" if rank == 1 else (f"@{rank}" if rank else "МИМО")
        print(f"  [{mark:>4}] {item['question'][:52]:<54} -> {docs[0]}")

    n = len(answerable)
    print(f"\nhit@1 = {hit1}/{n}, hit@3 = {hit3}/{n}, MRR = {mrr_sum/n:.2f}")
else:
    print("[пропущено: Qdrant недоступен]")

У нас: **hit@1 = 9/10, MRR = 0.90** — и единственный промах ровно там, где ждали: канарейка «Упал прод» ушла в `meetings` (лексика, синонимы, модуль 2). Метрика превратила знание «лексический поиск слаб на синонимах» из впечатления в число, которое будет мозолить глаза в каждом отчёте, пока не подключим e5.

Дальше — этап 2, и там так просто не будет.

## 4. Судья и его калибровка

У генерации нет эталонной строки для `==`: ответ «До 100% первые 10 дней при стаже от года» и эталон «Компания доплачивает 100% оклада 10 дней, если работаешь больше года» — строки разные, смысл один. Сравнивать смыслы умеет только языковая модель — значит, судьёй будет LLM (`qwen2.5:3b`, та же, что отвечает; в проде судьёй часто берут модель посильнее).

Тут ловушка, в которую попадает большинство: **судья — это та же LLM, которая галлюцинирует**. Верить его вердиктам «на слово» — значит измерять один шум другим. Отсюда два правила этого урока.

**Правило 1. Судью калибруют до того, как ему верить.** Составляем пары, где вердикт известен заранее (верные утверждения и заведомо подложные — с искажёнными числами, выдуманными условиями), прогоняем судью и считаем его точность. Это метрика для метрики.

**Правило 2. Только бинарные вопросы.** Мы сначала попробовали судью со шкалой «верно / частично / неверно» — и он поплыл в обе стороны: в одной версии промпта ставил «частично» дословно верному ответу, в другой (после few-shot) — «верно» ответу, противоречащему эталону; суммарно 6/9 на калибровке. Маленькой модели по силам один бинарный вопрос за раз: «да или нет?». Поэтому:

- **строгий судья** для faithfulness: «утверждение *прямо подтверждается* фрагментом?» — строгость здесь фича, она ловит выдумки;
- **толерантный судья** для coverage: «ответ *доносит* эталонный факт, пусть другими словами?» — и обязательно с текстом вопроса: без него судья не поймёт, что «первые 10 дней» в ответе — это про болезнь.

И третья роль — **разборщик**: разбить ответ на атомарные утверждения (JSON-списком), чтобы строгому судье было что проверять поштучно. Вот все трое:

In [ ]:
def chat(messages, num_predict=400, temperature=0.0):
    """t=0: для оценки нужна воспроизводимость (урок 1.2 про температуру)."""
    response = httpx.post(f"{OLLAMA_URL}/api/chat", json={
        "model": MODEL, "messages": messages, "stream": False,
        "options": {"temperature": temperature, "num_predict": num_predict},
    }, timeout=300)
    response.raise_for_status()
    return response.json()["message"]["content"]


EXTRACT_SYSTEM = (
    "Ты разбираешь текст на атомарные факты. Верни JSON-массив коротких "
    "самостоятельных утверждений, по одному факту в каждом. Только JSON, "
    "без пояснений."
)
VERIFY_SYSTEM = (      # строгий судья (faithfulness)
    "Ты проверяешь факты. Дан фрагмент документа и утверждение. Ответь одним "
    "словом: «да» — если утверждение прямо подтверждается фрагментом, «нет» — "
    "если не подтверждается или противоречит."
)
COVERAGE_SYSTEM = (    # толерантный судья (coverage), с вопросом
    "Ты сверяешь смысл. Дан вопрос, ответ системы на него и один эталонный "
    "факт. Ответь одним словом: «да» — если ответ доносит этот факт "
    "(пусть другими словами, короче или опираясь на контекст вопроса), "
    "«нет» — если факт в ответе отсутствует или искажён."
)


def extract_claims(answer: str) -> list[str]:
    raw = chat([{"role": "system", "content": EXTRACT_SYSTEM},
                {"role": "user", "content": answer}], num_predict=300)
    cleaned = raw.strip().removeprefix("```json").removesuffix("```").strip()
    try:
        claims = json.loads(cleaned)
        return [c for c in claims if isinstance(c, str) and c.strip()]
    except json.JSONDecodeError:
        return []          # не распарсилось - честно вернём пусто, увидим в отчёте


def supported(claim: str, context: str) -> bool:
    raw = chat([{"role": "system", "content": VERIFY_SYSTEM},
                {"role": "user", "content": f"Фрагмент:\n{context}\n\nУтверждение: {claim}"}],
               num_predict=10)
    return raw.strip().lower().startswith("да")


def covered(question: str, fact: str, answer: str) -> bool:
    raw = chat([{"role": "system", "content": COVERAGE_SYSTEM},
                {"role": "user", "content":
                 f"Вопрос: {question}\nОтвет системы: {answer}\nЭталонный факт: {fact}"}],
               num_predict=10)
    return raw.strip().lower().startswith("да")


if OLLAMA_AVAILABLE:
    fragment = ("Больничный оплачивается по закону: размер зависит от стажа. "
                "Компания доплачивает до 100% оклада первые 10 дней болезни, "
                "если стаж в «Векторике» больше года. Работать во время "
                "больничного запрещено.")
    strict_pairs = [   # (утверждение, ожидаемый вердикт)
        ("Компания доплачивает до 100% оклада первые 10 дней болезни.", True),
        ("Доплата действует при стаже в компании больше года.",          True),
        ("Работать во время больничного запрещено.",                     True),
        ("Компания доплачивает до 150% оклада.",                         False),
        ("Доплата действует первые 30 дней болезни.",                    False),
        ("Доплату получают только сотрудники отдела продаж.",            False),
        ("Больничный можно продлевать до полугода.",                     False),
        ("Сотрудникам выдают MacBook Pro 14.",                           False),
    ]
    ok = sum(supported(claim, fragment) == expected for claim, expected in strict_pairs)
    print(f"калибровка строгого судьи:     {ok}/{len(strict_pairs)}")

    q_bol = "Сколько компания доплачивает в первые дни болезни?"
    q_night = "За сколько минут дежурный должен отреагировать ночью?"
    tolerant_triples = [   # (вопрос, ответ системы, эталонный факт, ожидание)
        (q_bol, "До 100% оклада первые 10 дней, если стаж больше года. [1]",
         "Доплата — до 100% оклада", True),
        (q_bol, "До 100% оклада первые 10 дней, если стаж больше года. [1]",
         "Доплата действует первые 10 дней болезни", True),
        (q_bol, "До 100% оклада первые 10 дней, если стаж больше года. [1]",
         "Условие — стаж в компании больше года", True),
        (q_bol, "Компания доплачивает до 100% оклада. [1]",
         "Доплата действует первые 10 дней болезни", False),
        (q_bol, "В базе знаний нет ответа на этот вопрос.",
         "Доплата — до 100% оклада", False),
        (q_bol, "Зависит от региона и условий трудового договора.",
         "Доплата — до 100% оклада", False),
        (q_night, "Дежурный должен отреагировать в течение часа.",
         "Ночью дежурный должен отреагировать в течение 30 минут", False),
        (q_night, "15 минут днём и 30 минут ночью. [1]",
         "Ночью дежурный должен отреагировать в течение 30 минут", True),
        ("Сколько дней отпуска положено в году?", "Отпуск 28 календарных дней в год. [1]",
         "Положено 28 календарных дней отпуска в год", True),
        ("Сколько дней в году можно работать из-за границы?",
         "Работать из-за границы можно до 90 дней в году. [1]",
         "Нужно уведомить HR минимум за 2 недели", False),
    ]
    results = [(covered(q, fact, ans), expected, fact)
               for q, ans, fact, expected in tolerant_triples]
    ok = sum(got == expected for got, expected, _ in results)
    print(f"калибровка толерантного судьи: {ok}/{len(tolerant_triples)}")
    for got, expected, fact in results:
        if got != expected:
            print(f"  ошибка: «{fact}» -> {got} (ждали {expected})")
else:
    print("[пропущено: Ollama недоступна]")

В нашем прогоне: строгий судья — **8/8**, толерантный — **9/10**. Единственная ошибка — ложный «нет» на факте «Условие — стаж в компании больше года», который в ответе дословно есть. Три вывода из этих чисел:

1. **Ошибка — в сторону строгости.** Судья недооценивает, а не переоценивает: реальные скоры будут слегка занижены. Это удачное направление ошибки — для контроля качества лучше строгий педант, чем добрый попуститель.
2. **Сравнивать можно, абсолютам не верить.** Судья с точностью ~0.9 отлично видит разницу «конфигурация A против B» на десятке вопросов, но «faithfulness 0.83» против «0.85» — это шум.
3. **Не подгоняйте промпт судьи до 10/10.** Соблазн докрутить формулировку, пока калибровка не станет идеальной, велик — но так вы переобучите судью под калибровочный набор, и он перестанет измерять. Мы попробовали три формулировки, взяли лучшую и остановились (что делать с систематической ошибкой — в упражнении 3).

## 5. Этап 2 — генератор: faithfulness, coverage, отказы

Теперь собираем метрики поверх судей. На каждый вопрос golden-датасета генерируем ответ тем же конвейером, что в 3.1 (t=0 — для воспроизводимости), и считаем три вещи:

```text
                    ┌──────────── ответ системы ────────────┐
                    │                                        │
   разборщик: ответ -> [утверждение 1, утверждение 2, ...]   │
   строгий судья:      каждое подтверждено контекстом?       │   <- faithfulness
                                                             │      (правдивость)
   толерантный судья:  каждый эталонный факт донесён?  <-----┘   <- coverage
                                                                    (полнота)
   сравнение строк:    ответ - каноничный отказ?                 <- отказы
```

- **faithfulness** = доля утверждений ответа, подтверждённых найденным контекстом. Ловит выдумки. Считается «относительно контекста» — даже неполного: если ретривер промахнулся, а модель честно пересказала мусор, faithfulness высокий (виноват этап 1, и мы это увидим по hit@k).
- **coverage** = доля `reference_facts`, донесённых ответом. Ловит недосказанность. Правдивый ответ может быть куцым — эти оси **независимы**, и вы увидите это в отчёте прямо сейчас.
- **отказы** = на неотвечаемых вопросах ответ обязан быть каноничной фразой «В базе знаний нет ответа...» (правило 2 промпта из 3.1); на отвечаемых отказ — ложная тревога.

In [ ]:
SYSTEM_RAG = (      # дословно из урока 3.1
    "Ты — ассистент по внутренней базе знаний компании «Векторика».\n"
    "Правила:\n"
    "1. Отвечай ТОЛЬКО на основе фрагментов из блока <context>. "
    "Не используй никакие другие знания.\n"
    "2. Если в контексте нет ответа на вопрос — честно скажи: "
    "«В базе знаний нет ответа на этот вопрос».\n"
    "3. В конце ответа укажи использованные источники в виде [номер].\n"
    "4. Отвечай кратко и по делу, по-русски."
)


def rag_answer(question: str, system: str = SYSTEM_RAG) -> tuple[str, list[dict]]:
    found = retrieve(question)
    context = "\n\n".join(f"[{i}] {c['title']}:\n{c['text']}"
                          for i, c in enumerate(found, 1))
    answer = chat([{"role": "system", "content": system},
                   {"role": "user", "content":
                    f"<context>\n{context}\n</context>\n\nВопрос: {question}"}])
    return answer, found


def is_refusal(answer: str) -> bool:
    return "нет ответа" in answer.lower()


def eval_generation(item: dict, system: str = SYSTEM_RAG) -> dict:
    """Все метрики генератора для одного вопроса golden-датасета."""
    answer, found = rag_answer(item["question"], system)
    context = "\n".join(c["text"] for c in found)
    refused = is_refusal(answer)
    faith = cov = None
    if item["answerable"] and not refused:
        claims = extract_claims(answer)
        if claims:
            faith = sum(supported(c, context) for c in claims) / len(claims)
        cov = (sum(covered(item["question"], f, answer)
                   for f in item["reference_facts"])
               / len(item["reference_facts"]))
    elif item["answerable"] and refused:
        cov = 0.0          # ложный отказ - эталонные факты не донесены
    return {"id": item["id"], "answer": answer, "refused": refused,
            "faith": faith, "cov": cov}


def run_eval(system: str = SYSTEM_RAG) -> list[dict]:
    rows = []
    for item in golden:
        row = eval_generation(item, system)
        f = f"{row['faith']:.2f}" if row["faith"] is not None else "  - "
        c = f"{row['cov']:.2f}" if row["cov"] is not None else "  - "
        kind = "отказ" if row["refused"] else "ответ"
        print(f"  {row['id']:<20} {kind}  faith={f} cover={c}  «{row['answer'][:55]}»")
        rows.append(row)
    return rows


def summary(rows: list[dict]) -> dict:
    by_id = {g["id"]: g for g in golden}
    ans = [r for r in rows if by_id[r["id"]]["answerable"]]
    unans = [r for r in rows if not by_id[r["id"]]["answerable"]]
    faiths = [r["faith"] for r in ans if r["faith"] is not None]
    covs = [r["cov"] for r in ans if r["cov"] is not None]
    return {
        "faithfulness": sum(faiths) / len(faiths) if faiths else 0.0,
        "coverage": sum(covs) / len(covs) if covs else 0.0,
        "ложные отказы": f"{sum(r['refused'] for r in ans)}/{len(ans)}",
        "верные отказы": f"{sum(r['refused'] for r in unans)}/{len(unans)}",
    }


if QDRANT_AVAILABLE and OLLAMA_AVAILABLE:
    t0 = time.perf_counter()
    baseline_rows = run_eval()
    baseline = summary(baseline_rows)
    print(f"\nИТОГО: " + ", ".join(f"{k} {v if isinstance(v, str) else round(v, 2)}"
                                    for k, v in baseline.items()))
    print(f"время прогона: {time.perf_counter() - t0:.0f}с")
else:
    print("[пропущено: нужны Qdrant и Ollama]")

## 6. Читаем отчёт

Вот один из наших прогонов (ваши цифры будут немного другими — почему, разберём прямо сейчас):

```text
otpusk-days          ответ  faith=1.00 cover=1.00  «28 дней [1]»
bolnichny-doplata    ответ  faith=1.00 cover=0.33  «100% оклада [1]»
zagranitsa           ответ  faith=1.00 cover=0.50  «Из-за границы работа разрешена до 90 дней...»
upal-prod            ответ  faith=0.25 cover=0.00  «В первую очередь необходимо провести анализ...»
overtime-weekend     отказ                          «В базе знаний нет ответа...»
...
ИТОГО: faithfulness 0.88, coverage 0.68, ложные отказы 0/10, верные отказы 3/3
```

Четыре сюжета, ради которых всё строилось:

**1. `zagranitsa` — мерцающая галлюцинация.** В прогоне выше всё чинно: «до 90 дней», как в документе. Но при подготовке урока **этот же вопрос, с тем же промптом и t=0, трижды подряд дал «56 дней»** — число не из документа, не из вопроса, ниоткуда (faith и cover — по нулям, метрики поймали мгновенно). Оба исхода настоящие, и это важнее самой галлюцинации: **даже t=0 не даёт бит-в-бит воспроизводимости на GPU** — логиты чуть плавают от прогона к прогону, и на пограничных токенах жадный выбор перещёлкивается. Выводы: галлюцинации бывают *мерцающими* — единичный прогон их не исключает, только регулярные; и сравнивать между конфигурациями надо агрегаты по датасету, а не единичные ответы.

**2. `upal-prod` — этапы разделили вину.** Ретривер промахнулся (мы это уже знаем из hit@k), контекст пришёл мусорный — и модель **нарушила grounding**: вместо отказа выдала общие советы из своих весов. Низкий faithfulness показывает проблему, hit@k говорит, с какого этапа начинать чинить, — а заодно мы узнали, что правило «отвечай только по контексту» под давлением вопроса не железное. Это тема урока 3.4.

**3. `bolnichny-doplata` — оси действительно независимы.** Ответ «100% оклада» правдив (faith 1.00 — это есть в контексте), но донёс 1 факт из 3 (cover 0.33): потерялись «первые 10 дней» и «стаж от года». Пользователь с ответом «100% оклада» уйдёт планировать бюджет неверно — при формально безупречной правдивости. Промпт просил «кратко и по делу» — вот и результат; вернёмся к этому в секции 8.

**4. Отказы — 3/3 и ни одной ложной тревоги.** Включая коварный «переработки в выходные» рядом с найденными надбавками за дежурство.

Числа `0.88 / 0.68` сами по себе не «хорошо» и не «плохо» — это **база для сравнения**. Смысл они приобретут в секции 8, когда мы что-нибудь изменим.

## 7. Answer relevancy: идея обратных вопросов

Ещё одна ось: ответ может быть правдивым, полным... и не на тот вопрос («Как оформить отпуск?» — «Отпуск 28 дней»). У RAGAS для этого есть красивый приём: **сгенерировать из ответа вопросы, на которые он отвечает**, и сравнить их с исходным вопросом эмбеддингами. Отвечал не на то — обратные вопросы уедут в сторону, и косинус это покажет.

Эмбеддинги у нас — TF-IDF из 3.2 (e5 по-прежнему опциональна), поэтому сравнение лексическое: парафразы оно недооценит, но грубый уход от вопроса ловит уверенно. Демонстрация на паре «ответ по делу / ответ-уход»:

In [ ]:
REVQ_SYSTEM = (
    "Дан ответ. Придумай 3 коротких вопроса, на которые этот ответ отвечает "
    "напрямую. Верни JSON-массив из 3 строк, без пояснений."
)


def relevancy(question: str, answer: str) -> float | None:
    raw = chat([{"role": "system", "content": REVQ_SYSTEM},
                {"role": "user", "content": answer}], num_predict=150)
    # модель иногда выдаёт битый JSON (проверено на себе) - поэтому не json.loads,
    # а регулярка: выуживаем строки в кавычках, JSON цел или нет - неважно
    reverse_questions = re.findall(r'"([^"\n]{10,200})"', raw)[:3]
    if not reverse_questions:
        print("    [обратные вопросы не распарсились - пропуск]")
        return None
    q_vec = emb.embed_query(question)
    sims = []
    for rq in reverse_questions:
        rq_vec = emb.embed_query(rq)
        sim = sum(a * b for a, b in zip(q_vec, rq_vec))   # оба уже L2-нормированы
        sims.append(sim)
        print(f"    [{sim:.2f}] {rq}")
    return sum(sims) / len(sims)


if QDRANT_AVAILABLE and OLLAMA_AVAILABLE:
    def show_relevancy(label: str, answer: str) -> None:
        print(label)
        score = relevancy(question, answer)
        if score is not None:
            print(f"  relevancy = {score:.2f}\n")

    question = "Сколько дней отпуска положено в году?"
    print(f"вопрос: {question}\n")
    show_relevancy("ответ ПО ДЕЛУ: «Каждому сотруднику положено 28 календарных дней...»",
        "Каждому сотруднику положено 28 календарных дней оплачиваемого отпуска в год.")
    show_relevancy("ответ-УХОД: «Отпуск оформляется через HR-портал...»",
        "Отпуск оформляется через HR-портал, заявку согласует тимлид и затем HR.")
else:
    print("[пропущено: нужны Qdrant и Ollama]")

У нас: по делу — **0.43**, уход — **0.15**. Абсолютные значения скромные (TF-IDF наказывает за любые синонимы в обратных вопросах), но разрыв в 3 раза виден отлично. С e5 те же числа были бы ближе к 0.9 / 0.3.

## 8. RAGAS: фреймворк и его сюрприз

Всё, что мы построили руками, существует в виде библиотеки — [RAGAS](https://docs.ragas.io). Соответствие почти дословное:

| наша метрика | в RAGAS |
|---|---|
| faithfulness (разборщик + строгий судья) | `Faithfulness` |
| coverage (факты эталона в ответе) | семейство `AnswerCorrectness` / `ContextRecall` |
| relevancy (обратные вопросы + косинус) | `AnswerRelevancy` |
| hit@k, MRR | не нужен LLM — RAGAS тут ни к чему |

Зачем тогда был весь урок? Затем, что RAGAS — это **те же судьи с теми же слабостями**, только упакованные в чужие промпты, которые вы не калибровали. Сейчас это выстрелит.

Установка — с граблями зависимостей (мы наступили, вам не обязательно): свежий `ragas` требует старый `langchain`, и наоборот. Рабочая связка августа 2026:

```bash
pip install "ragas==0.2.15" "langchain-community<0.4" "langchain-ollama<0.3" "langchain<1"
```

Подключение локального стека — через обёртки: `LangchainLLMWrapper(ChatOllama(...))` для судьи и `LangchainEmbeddingsWrapper` для эмбеддингов. Причём эмбеддинги мы отдадим **свои TF-IDF**, обернув их в интерфейс LangChain, — контракт из урока 3.2 сработает и здесь.

Эксперимент — два хрустально чистых сэмпла, на которых вердикт очевиден даже без машины:

1. **идеальный ответ**: дословно подтверждён контекстом, совпадает с эталоном;
2. **наглая выдумка**: «45 дней отпуска и 13-я зарплата» при контексте «28 дней».

Идеальный должен получить faithfulness ~1.0, выдумка ~0.0. Проверяем:

In [ ]:
if RAGAS_AVAILABLE and OLLAMA_AVAILABLE:
    from langchain_core.embeddings import Embeddings
    from langchain_ollama import ChatOllama

    from ragas import EvaluationDataset, SingleTurnSample, evaluate
    from ragas.embeddings import LangchainEmbeddingsWrapper
    from ragas.llms import LangchainLLMWrapper
    from ragas.metrics import AnswerRelevancy, Faithfulness
    from ragas.run_config import RunConfig

    class TfidfLangchainEmbeddings(Embeddings):
        """Наш TF-IDF из 3.2 в интерфейсе LangChain - никаких скачиваний."""

        def embed_documents(self, texts):
            return emb.embed_docs(texts)

        def embed_query(self, text):
            return emb.embed_query(text)

    judge = LangchainLLMWrapper(ChatOllama(model=MODEL, temperature=0.0,
                                           base_url=OLLAMA_URL))
    wrapped_emb = LangchainEmbeddingsWrapper(TfidfLangchainEmbeddings())

    samples_ru = [
        SingleTurnSample(       # идеальный ответ
            user_input="Сколько компания доплачивает в первые дни болезни?",
            response="Компания доплачивает до 100% оклада первые 10 дней "
                     "болезни, если стаж больше года. [1]",
            retrieved_contexts=[
                "Больничный оплачивается по закону: размер зависит от стажа. "
                "Компания доплачивает до 100% оклада первые 10 дней болезни, "
                "если стаж в «Векторике» больше года."],
            reference="Доплата до 100% оклада первые 10 дней болезни при стаже от года.",
        ),
        SingleTurnSample(       # наглая выдумка
            user_input="Сколько дней отпуска положено в году?",
            response="Сотрудникам положено 45 дней отпуска, а также 13-я зарплата.",
            retrieved_contexts=[
                "Каждому сотруднику положено 28 календарных дней оплачиваемого "
                "отпуска в год. Отпуск можно дробить на части."],
            reference="28 календарных дней оплачиваемого отпуска в год.",
        ),
    ]

    t0 = time.perf_counter()
    result_ru = evaluate(EvaluationDataset(samples=samples_ru),
                         metrics=[Faithfulness(llm=judge),
                                  AnswerRelevancy(llm=judge, embeddings=wrapped_emb)],
                         run_config=RunConfig(timeout=300, max_workers=1),
                         show_progress=False)
    print(f"RAGAS на русских сэмплах ({time.perf_counter()-t0:.0f}с):")
    print(result_ru)
    print(result_ru.to_pandas()[["faithfulness", "answer_relevancy"]]
          .rename(index={0: "идеальный ответ", 1: "наглая выдумка"}).round(2))
else:
    print("[пропущено: нужен ragas (pip install см. выше) и Ollama]")

У нас вышло:

```text
                 faithfulness  answer_relevancy
идеальный ответ           0.0              0.37
наглая выдумка            0.0               NaN
```

**Ноль. Идеальному ответу.** Тому самому, которому наш калиброванный судья ставит 1.00. И выдумке — тот же ноль: метрика перестала различать хорошее и плохое, а это единственное, зачем она существует. Хуже того, от прогона к прогону числа гуляют: в наших запусках идеальный ответ получал то 0.00, то 0.67 — как повезёт с парсингом вердиктов — и ни разу заслуженную ~1.0. Числу, которое так себя ведёт, нельзя доверить ни алерт, ни сравнение конфигураций.

## 9. Разбор: почему фреймворк молча выдал мусор

Ошибки нет — есть *несовместимое сочетание*. Внутри RAGAS та же схема, что у нас: разбить ответ на утверждения → проверить каждое → распарсить вердикты. Но:

- **промпты RAGAS — английские и многошаговые**, с длинными инструкциями и примерами;
- ответ судьи парсится **строгим JSON-парсером** (Pydantic-схемы);
- наш судья — **3B-модель**, которой английская инструкция поверх русского контента уже тяжела; она отвечает «почти в формате», парсер молча получает пусто;
- пустой список утверждений → метрика 0.0 **без единого предупреждения**.

Каждое звено разумно, вместе — тихий ноль. Проверим диагноз: если дело в языковом разрыве, на английских сэмплах должно полегчать.

In [ ]:
if RAGAS_AVAILABLE and OLLAMA_AVAILABLE:
    samples_en = [
        SingleTurnSample(       # идеальный ответ, теперь по-английски
            user_input="How much does the company pay during the first days of sick leave?",
            response="The company tops up to 100% of salary for the first 10 days "
                     "of sick leave, if tenure is over one year. [1]",
            retrieved_contexts=[
                "Sick leave is paid according to the law. The company tops up to "
                "100% of salary for the first 10 days of sick leave if the "
                "employee's tenure at Vectorika is over one year."],
        ),
        SingleTurnSample(       # выдумка
            user_input="How many vacation days are employees entitled to per year?",
            response="Employees get 45 vacation days and a 13th salary bonus.",
            retrieved_contexts=[
                "Every employee is entitled to 28 calendar days of paid vacation "
                "per year. Vacation can be split into parts."],
        ),
    ]
    t0 = time.perf_counter()
    result_en = evaluate(EvaluationDataset(samples=samples_en),
                         metrics=[Faithfulness(llm=judge)],
                         run_config=RunConfig(timeout=300, max_workers=1),
                         show_progress=False)
    print(f"RAGAS на английских сэмплах ({time.perf_counter()-t0:.0f}с):")
    print(result_en)
else:
    print("[пропущено: нужен ragas и Ollama]")

У нас на английском выходило `faithfulness 0.33–0.50` в разных прогонах — машинерия **частично ожила** (утверждения стали извлекаться), но идеальный ответ всё равно не получил 1.0: многошаговым промптам RAGAS нужен судья покрупнее. Диагноз подтверждён по обеим осям: язык + размер судьи.

Что из этого следует — и что не следует:

- **RAGAS не сломан.** С сильным судьёй (облачная модель или локальная 70B+) и английским контентом — это отличный, продуманный инструмент, и его метрики — ровно те идеи, которые мы построили руками.
- **Фреймворк не освобождает от калибровки.** Он *переносит* судью в чужие промпты, которые вы не видели. Правило то же: прежде чем верить числам, скормите системе пары с заранее известным вердиктом — как мы сделали с «хрустальными» сэмплами. Это 10 минут, и именно они отделяют измерение от самообмана.
- **Мораль нашей истории.** Начни мы урок с RAGAS — получили бы 0.0, решили бы, что RAG безнадёжен, и «чинили» бы исправную систему. Свои метрики с откалиброванным судьёй = полный контроль над каждым звеном: промпты русские, парсинг наш, ошибки судьи измерены.

## 10. Метрики как регрессионные тесты

Финальный акт — то, ради чего метрики существуют. Помните `bolnichny-doplata` с cover 0.33? Ответы куцые, потому что промпт просит «кратко и по делу». Очевидное «улучшение»: попросим отвечать полно. Интуиция кричит, что станет лучше. Проверим прогоном — **меняем одну переменную** (правило 4 промпта, урок 1.7) и гоняем тот же golden:

In [ ]:
SYSTEM_RAG_V2 = SYSTEM_RAG.replace(
    "4. Отвечай кратко и по делу, по-русски.",
    "4. Отвечай одним полным предложением, включая ВСЕ условия, сроки и "
    "цифры, которые контекст сообщает по теме вопроса. По-русски.",
)

if QDRANT_AVAILABLE and OLLAMA_AVAILABLE:
    print("прогон с промптом v2 («отвечай полно»):")
    v2_rows = run_eval(SYSTEM_RAG_V2)
    v2 = summary(v2_rows)
    print("\nСРАВНЕНИЕ:")
    print(f"{'метрика':<16} {'v1 (кратко)':<14} v2 (полно)")
    for key in baseline:
        b, n = baseline[key], v2[key]
        b_str = b if isinstance(b, str) else f"{b:.2f}"
        n_str = n if isinstance(n, str) else f"{n:.2f}"
        print(f"{key:<16} {b_str:<14} {n_str}")
else:
    print("[пропущено: нужны Qdrant и Ollama]")

Наш результат:

```text
метрика          v1 (кратко)    v2 (полно)
faithfulness     0.88           0.93
coverage         0.68           0.77
ложные отказы    0/10           0/10
верные отказы    3/3            2/3   <- !!!
```

Полнота выросла, правдивость выросла. Улучшение? **Смотрим на последнюю строку: отказы просели.** Просьба «отвечай полно, включая ВСЁ» конкурирует с правом не знать — и на неотвечаемых вопросах модель начинает выкручиваться. В разных наших прогонах v2 проваливал одну-две ловушки из трёх (та самая нестабильность из секции 6 — ещё одна причина держать в golden несколько ловушек, а не одну). Разберём провалы глазами — метрика говорит *куда* смотреть, но не *что* там:

- `overtime-weekend` — **стабильный неудачник** (проваливался в каждом нашем v2-прогоне): «За ночные вызовы компенсируется отгул на следующий день» — а спрашивали про **переработки в выходные**. Модель подменила вопрос соседним, на который ответ в контексте есть. Это **настоящая регрессия**, самый опасный вид галлюцинации: каждое слово правдиво, ответ в целом — ложь.
- `parking` — в одном из прогонов ответил «В базе знаний нет **информации** о наличии парковки...»: модель честно отказалась, но **другими словами**, и наш детектор-строка (`"нет ответа" in answer`) её не узнал. Это ложная тревога — сломался детектор, а не система. Урок: метрика-строка хрупка к переформулировкам; детектор стоит расширить или сделать судейским.

Вот и вся суть урока в одной таблице: «очевидное улучшение» оказалось **трейдоффом**, который без прогона уехал бы в прод. Полнота и дисциплина отказов конфликтуют — балансировать их на глаз невозможно, по метрикам — рутина (в упражнении 2 вы найдёте формулировку, которая забирает и то и другое).

## ⚠️ Частые ошибки

| ошибка | чем оборачивается | лекарство |
|---|---|---|
| верить судье без калибровки | измеряете шум шумом; RAGAS поставил 0.0 идеальному ответу — а без калибровки вы бы поверили | подложные пары с известным вердиктом, точность судьи — публичное число |
| шкалы («от 1 до 10», «верно/частично/неверно») у маленького судьи | вердикты плавают в обе стороны (у нас: 6/9) | раскладывать на бинарные «да/нет» |
| подгонять промпт судьи до 10/10 на калибровке | судья переобучен под калибровочный набор и больше ничего не меряет | взять лучшее из 2–3 формулировок и остановиться |
| в golden только отвечаемые вопросы | «отвечает на всё» выглядит как успех, хотя это худший диагноз | неотвечаемые ловушки, особенно тематически близкие |
| мерить только faithfulness (или только coverage) | правдиво-но-куце и полно-но-выдумано неразличимы | обе оси: они независимы (faith 1.00 / cover 0.33 — реальный случай) |
| нет вопроса-канарейки на известную слабость | слабость забывается до первого инцидента | «Упал прод» живёт в golden, пока не подключим e5 |
| детектор отказа = точная строка | перефразированный отказ считается ответом (наш `parking`) | список паттернов или судейская проверка «это отказ?» |
| выводы по 2–3 сэмплам | одно везение меняет «метрику» на треть | 10+ вопросов на каждый срез, тренды — по десяткам |
| поменяли промпт и топ-к сразу | непонятно, что подействовало | одна переменная за раз (урок 1.7) |
| считать, что t=0 даёт бит-в-бит воспроизводимость | на GPU логиты плавают, пограничные токены перещёлкиваются между прогонами (наша «мерцающая» галлюцинация) | воспроизводимость искать в агрегатах по датасету, а не в единичных ответах |
| сравнивать абсолютные скоры разных судей | у каждого судьи своя строгость | сравнения валидны только внутри одного судьи |

## 🏋️ Упражнения

**1. Расширьте golden.** Добавьте вопрос «За сколько минут откатывается неудачный деплой в production?» (ответ есть в `09-deploy.txt`): составьте запись с `expected_docs` и атомарными `reference_facts`, прогоните через `retrieve` и `eval_generation`. Попадёт ли ретривер? Донесёт ли модель факт?

**2. Почините трейдофф.** Промпт v1 дисциплинированно отказывается, но отвечает куце; v2 — полно, но отказы поплыли. Сформулируйте правило, которое даст **и** полноту, **и** отказы, и докажите прогоном: полнота на бывших куцых вопросах + все три отказа на ловушках. Подсказка: попробуйте развести «есть ли прямой ответ?» и «как отвечать, если есть» по разным правилам.

**3. Спасите факт от судьи.** На калибровке толерантный судья систематически валит факт «Условие — стаж в компании больше года» (ложный «нет»). Popular wisdom говорит «прогоните трижды с t=0.7 и возьмите большинство». Проверьте, спасает ли голосование — и если нет, найдите, что *на самом деле* чинит вердикт.

## 📝 Мини-квиз

<details><summary><b>1. Зачем оценивать ретривер и генератор раздельно, если пользователь видит только финальный ответ?</b></summary>

Для диагноза. Плохой финальный ответ при промахе ретривера — вина поиска (чинить чанкинг/эмбеддер/индекс), при хорошем ретривале — вина генерации (чинить промпт/модель). Наш `upal-prod` показал это в лицах: hit@k нашёл виновника, faithfulness — последствия. Без разделения вы будете чинить не тот этап.
</details>

<details><summary><b>2. У ответа faithfulness 1.00 и coverage 0.33. Что это значит и опасен ли такой ответ?</b></summary>

Каждое утверждение ответа подтверждено контекстом (ничего не выдумано), но донесена лишь треть эталонных фактов — ответ правдивый и куцый. Опасен: «доплата 100% оклада» без «первых 10 дней» и «стажа от года» приведёт пользователя к неверным выводам, хотя формально система не соврала.
</details>

<details><summary><b>3. Зачем в golden-датасете вопросы, на которые в базе нет ответа?</b></summary>

Чтобы измерять дисциплину отказов. Система, отвечающая на всё, не «полезнее» — она выдумывает, просто вы этого не видите. Самые ценные ловушки — тематически близкие («переработки в выходные» рядом с надбавками за дежурство): именно на них модель подменяет вопрос соседним, как показал наш v2-прогон.
</details>

<details><summary><b>4. Судья показал 9/10 на калибровке, единственная ошибка — ложный «нет». Как пользоваться его вердиктами?</b></summary>

Скоры слегка занижены (судья строг), поэтому: сравнивать конфигурации между собой — можно и нужно; трактовать разницу в пару сотых — нельзя, это шум; выжимать 10/10 правкой промпта — не стоит, переобучите судью под калибровочный набор. Точность судьи указывают рядом с метриками.
</details>

<details><summary><b>5. RAGAS выдал 0.0 на ответе, который заведомо идеален. Какие три подозреваемых проверить первыми?</b></summary>

(1) Судья слишком мал для многошаговых промптов фреймворка; (2) язык: промпты RAGAS английские, контент русский; (3) парсинг: строгий JSON-парсер молча превращает «почти формат» в пустоту, а пустоту — в 0.0. Проверка — хрустальные сэмплы с известным вердиктом: тот же принцип калибровки, что и для собственного судьи.
</details>

<details><summary><b>6. Изменение промпта заметно подняло coverage. Можно деплоить?</b></summary>

Рано: смотреть надо на весь отчёт, а не на улучшившуюся метрику. У нас при таком улучшении просели отказы — и одна из «регрессий» оказалась настоящей (подмена вопроса соседним), а другая — ложной тревогой детектора. Правило: любое изменение → полный прогон golden → разбор регрессий глазами.
</details>

## Решения

**Решение 1 — новый вопрос в golden.** Запись собираем по анатомии из секции 2; факт в `09-deploy.txt` один: «Откат (rollback) — одной кнопкой в CI, возвращает предыдущий образ за 2–3 минуты».

In [ ]:
if QDRANT_AVAILABLE and OLLAMA_AVAILABLE:
    new_item = {
        "id": "rollback-time",
        "question": "За сколько минут откатывается неудачный деплой в production?",
        "expected_docs": ["deploy"],
        "answerable": True,
        "reference": "Откат выполняется одной кнопкой в CI и возвращает "
                     "предыдущий образ за 2-3 минуты.",
        "reference_facts": ["Откат возвращает предыдущую версию за 2-3 минуты"],
    }
    docs = [c["doc_id"] for c in retrieve(new_item["question"])]
    rank = next((i for i, d in enumerate(docs, 1)
                 if d in new_item["expected_docs"]), None)
    print(f"ретривер: {docs} -> " + (f"hit@{rank}" if rank else "МИМО"))

    row = eval_generation(new_item)
    print(f"ответ: «{row['answer']}»")
    print(f"faithfulness={row['faith']}, coverage={row['cov']}")
else:
    print("[пропущено: нужны Qdrant и Ollama]")

**Решение 2 — правило, которое забирает и полноту, и отказы.** Трюк в том, чтобы **развести решение и стиль по разным правилам**: сначала модель решает, есть ли прямой ответ именно на этот вопрос (и если нет — жёсткая каноничная фраза и стоп), и только потом — как отвечать полно. В v2 эти две инструкции конкурировали внутри одного правила «включая ВСЁ» — и полнота победила честность.

In [ ]:
SYSTEM_RAG_V3 = (
    "Ты — ассистент по внутренней базе знаний компании «Векторика».\n"
    "Правила:\n"
    "1. Отвечай ТОЛЬКО на основе фрагментов из блока <context>.\n"
    "2. Сначала реши: есть ли в контексте ПРЯМОЙ ответ именно на этот вопрос "
    "(а не на похожий). Если нет — ответь ровно одной фразой: "
    "«В базе знаний нет ответа на этот вопрос» и больше ничего не пиши.\n"
    "3. Если ответ есть — дай его одним полным предложением, включая все "
    "условия, сроки и цифры из контекста по теме вопроса.\n"
    "4. В конце укажи использованные источники в виде [номер]. По-русски."
)

if QDRANT_AVAILABLE and OLLAMA_AVAILABLE:
    print("ловушки (ждём три отказа):")
    for item in golden:
        if not item["answerable"]:
            answer, _ = rag_answer(item["question"], SYSTEM_RAG_V3)
            mark = "отказ" if is_refusal(answer) else "ОТВЕТИЛ!"
            print(f"  [{mark}] «{answer[:65]}»")

    print("\nбывшие куцые (ждём полноту):")
    for qid in ("bolnichny-doplata", "zagranitsa"):
        item = next(g for g in golden if g["id"] == qid)
        row = eval_generation(item, SYSTEM_RAG_V3)
        print(f"  cover={row['cov']:.2f}  «{row['answer'][:65]}»")

    print("\nбонус - канарейка upal-prod (v1 отвечал из весов):")
    answer, _ = rag_answer("Упал прод, что делать в первую очередь?", SYSTEM_RAG_V3)
    print(f"  «{answer[:70]}»")
else:
    print("[пропущено: нужны Qdrant и Ollama]")

У нас v3 вернул **все три отказа**, полноту не уронил (доплата — два-три факта из трёх в разных прогонах, из-за границы — честные «до 90 дней») и дал неожиданный бонус: на канарейке `upal-prod` с мусорным контекстом модель теперь **отказывается**, а не сочиняет из весов — правило «прямой ответ именно на этот вопрос (а не на похожий)» закрыло и эту дыру. Заметьте: мы не «почувствовали», что v3 лучше, — мы это *измерили*.

**Решение 3 — спасение факта от судьи.**

In [ ]:
if OLLAMA_AVAILABLE:
    question = "Сколько компания доплачивает в первые дни болезни?"
    answer = "До 100% оклада первые 10 дней, если стаж больше года. [1]"
    fact = "Условие — стаж в компании больше года"

    def covered_hot(q, f, a):     # тот же судья, но t=0.7 - для голосования
        raw = chat([{"role": "system", "content": COVERAGE_SYSTEM},
                    {"role": "user", "content":
                     f"Вопрос: {q}\nОтвет системы: {a}\nЭталонный факт: {f}"}],
                   num_predict=10, temperature=0.7)
        return raw.strip().lower().startswith("да")

    print(f"один вызов (t=0):      {covered(question, fact, answer)}")
    votes = [covered_hot(question, fact, answer) for _ in range(3)]
    print(f"голосование 3× (t=0.7): {votes} -> большинство: {sum(votes) >= 2}")

    fact_full = "Доплата до 100% оклада положена при стаже больше года"
    print(f"\nфакт с главным утверждением: «{fact_full}»")
    verdicts = [covered(question, fact_full, answer) for _ in range(3)]
    print(f"три вызова (t=0):      {verdicts}")
else:
    print("[пропущено: Ollama недоступна]")

У нас голосование выдало `[False, False, False]` — **не спасло**. И это важный отрицательный результат: голосование лечит *шум* (нестабильные вердикты), а наша ошибка — *систематическая*: судья уверенно валит формулировку «Условие — ...». Систематические ошибки чинятся не усреднением, а устранением причины.

Причина находится экспериментально: судья не готов признать «условие» донесённым **в отрыве от главного утверждения**. Мы прогнали несколько формулировок по многу раз: обрывки вида «Условие — стаж...» и «Доплату получают сотрудники со стажем больше года» судья валит или крутит на грани (в наших прогонах один и тот же вариант получал то «да», то «нет» — снова пограничные токены). А факт, несущий главное утверждение вместе с условием, — «Доплата до 100% оклада положена при стаже больше года» — стабильно получает «да».

Практический вывод для golden-датасета: **эталонные факты — это тоже промпты**. «Атомарный» не значит «обрубленный»: каждый факт должен быть самостоятельным мини-утверждением — с подлежащим, сказуемым и главным смыслом, а не висящим в воздухе условием. И да — поправить формулировку факта в `golden.json` после такого разбора не стыдно: датасет — живой артефакт, главное, чтобы правка чинила формулировку, а не подгоняла ответ под систему.

## Итоги

- **Оценка RAG = оценка этапов.** Ретривер меряется бесплатно и детерминированно (hit@1 9/10, MRR 0.90, канарейка «Упал прод» — МИМО, как и положено лексике). Генератор — судьёй по golden-датасету.
- **Судье нельзя верить на слово.** Калибровка на подложных парах: строгий 8/8, толерантный 9/10, ошибки в сторону строгости. Сложные шкалы валятся (6/9) — только бинарные «да/нет». Голосование лечит шум, но не систематическую строгость.
- **Метрики поймали настоящую галлюцинацию** — «56 дней» вместо «до 90» при идеальном ретривале, причём *мерцающую*: t=0 на GPU не гарантирует повторяемости, единичный прогон — не истина. И настоящую регрессию: «улучшение» промпта подняло coverage, но уронило отказы (подмена вопроса соседним). Обе истории глазами не ловятся.
- **Оси качества независимы**: faith 1.00 / cover 0.33 — правдиво, но куце. Обе метрики обязательны.
- **RAGAS = те же идеи в чужих промптах.** На нашем стеке (3B-судья + русский контент) — 0.0 идеальному ответу: английские многошаговые промпты + строгий JSON-парсинг дают тихий ноль. С большим судьёй и английским контентом — рабочий инструмент. Калибровка обязательна для любого судьи — своего или чужого.

| приём урока | родом из |
|---|---|
| «метрики вместо впечатлений», одна переменная за раз | урок 1.7 |
| eval-вопросы «Векторики», канарейка про синонимы | уроки 2.5–2.7 |
| RAG-конвейер и grounding-правила | урок 3.1 |
| retrieve через Qdrant, TF-IDF-эмбеддер как интерфейс | урок 3.2 |
| t=0 для воспроизводимости, температура судьи | урок 1.2 |

**Что дальше.** Сегодняшний отчёт дважды показал одну и ту же дыру: модель *отвечает*, когда должна *отказаться* («упал прод» в v1, «переработки» в v2). Промптом мы её прикрыли (v3), но промпт — уговор, а не гарантия. Урок 3.4 строит защиту от галлюцинаций **до и после генерации**: пороги релевантности (отказ до вызова LLM — скоры у нас уже есть), обязательное цитирование и проверку ответов. Неотвечаемые вопросы из golden станут там основным полигоном.